# Lab 05: How to rebuild a RAG pipeline with LangChain

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | Demo**

## Goal

Rebuild the search of Labs 01 and 02 with LangChain and check it against your own, question by question. The moment
that teaches it: both searches find the right paragraph for all 30 questions, yet they return different chunks for
every one of them, because a LangChain default returns four chunks where your search returns three, and the overall
score cannot show it. **Compare a framework with your own code question by question, not by a total.** Parts E to H
(self-study) rebuild the answer step, a metadata filter and the module the next labs use, with the same comparison.

After Labs 01 to 04 you have a pipeline in plain Python that you can trace from the search to the cited answer. The
labs after this one use [LangChain](https://docs.langchain.com/oss/python/langchain/overview), a **framework** whose
components (documents, vector stores, chat models) do those steps for you and make some choices through their
defaults. This lab rebuilds the same pipeline with it and finds where the two differ before you measure anything on
it.

## What you will learn

- How to turn chunk records into LangChain documents that keep their IDs and fields.
- How LangChain's embeddings and chat model are built from `models.env`, and what the embeddings object sends to the
  API.
- How to index documents in a vector store and search them with scores.
- How to compare a framework's results with your own question by question, and find the default that made them
  differ.
- How to get a cited answer back as a Python object, check it with Lab 02's quote check, and explain why its answers
  can differ from your own.
- How to leave documents out of a search with a metadata filter.

**In class:** Parts A, C and D, shown as a demo, with the `lc_embeddings()` cell at the start of Part B, which
builds the embeddings Part C's store uses. The rest of Part B and Parts E to H are self-study.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-27
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, and `JSON_MODE` and `TOKEN_ENCODING` from
`models.env`. It makes about 110 short chat calls and embeds the corpus two or three times (about 21,000 tokens each),
well under one US cent on `gpt-4o-mini`. The saved outputs and every count in the text come from `gpt-4o-mini`; on
another model the counts can move. Model replies vary from run to run and similarity scores can move in the
third decimal, so compare the order of results and the kind of reply with the text, not every number.

In the printed output, *plain* is your pipeline from Labs 01 and 02 and *LangChain* the new one.

---

## Part A: Turn the chunks into documents

Part A gives LangChain the corpus in the shape its components take.

Load the same corpus as Lab 02: the 272 SQuAD chunks and the five product policies, each policy as one chunk whose ID
is its source ID.

In [3]:
# corpus: 277 records, the 272 chunks followed by the five policies; each record is a dictionary
import json

DATA = ROOT / "data"
chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
corpus = chunks + [{**policy, "chunk_id": policy["source_id"]} for policy in policies]
print(len(corpus), "records")
print("a chunk: ", list(corpus[0]))
print("a policy:", list(corpus[-1]))

277 records
a chunk:  ['chunk_id', 'source_id', 'title', 'start', 'end', 'text', 'tokens']
a policy: ['source_id', 'title', 'text', 'version', 'access', 'chunk_id']


- 277 records: 272 chunks plus 5 policies.
- A chunk holds its `text`, its own `chunk_id`, the `source_id` of its paragraph, where it sits in that paragraph
  (`start`, `end`) and its size in `tokens`.
- A policy holds `version` and `access` instead. `{**policy, "chunk_id": ...}` copies the policy's fields and adds a
  `chunk_id` equal to its `source_id` (H01 to H05), so every record can be found by the same key.

Every LangChain component that reads text takes a **`Document`**
([reference](https://reference.langchain.com/python/langchain-core/documents/base/Document)), LangChain's record for
one piece of text. It has three parts:

- `page_content`: the text a store embeds and a model reads.
- `metadata`: a dictionary for everything else.
- `id`: the key a store files it under.

Build one by hand from the first record:

In [4]:
# document: the first record as a Document; the text goes to page_content, every other field to metadata
from langchain_core.documents import Document

record = corpus[0]
document = Document(id=record["chunk_id"], page_content=record["text"],
                    metadata={name: value for name, value in record.items() if name != "text"})
print("id:      ", document.id)
print("metadata:", document.metadata)
print("text:    ", document.page_content[:80], "...")

id:       S00-P00-C00
metadata: {'chunk_id': 'S00-P00-C00', 'source_id': 'S00-P00', 'title': 'Super_Bowl_50', 'start': 0, 'end': 486, 'tokens': 98}
text:     Super Bowl 50 was an American football game to determine the champion of the Nat ...


The text moved to `page_content` and every other field to `metadata`. **The `id` matters most:** a store keys each
document by it, and a citation names it; a document without one gets a random ID from the store. The metadata keeps
what later steps read: `source_id` for hit@3, and on the policies `version` and `access`.

Now all 277, with the same line inside a list comprehension:

In [5]:
# documents: one Document per record, built the same way as the one above
documents = [Document(id=record["chunk_id"], page_content=record["text"],
                      metadata={name: value for name, value in record.items() if name != "text"})
             for record in corpus]
print(len(documents), "documents")
print(documents[-1])

277 documents
page_content='Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.' metadata={'source_id': 'H05', 'title': 'Atlas billing', 'version': '2026-10-01', 'access': 'public', 'chunk_id': 'H05'}


`documents[-1]` is the last policy, H05: its text in `page_content`, and its `source_id`, title, `version`, `access` and
`chunk_id` in `metadata`.

> A `Document` is a text plus a dictionary of what you need to know about it. LangChain's stores and models read the
> text; **your checks read the metadata**.

### Checkpoint A

| You should see | Our run |
|---|---|
| Corpus | 277 records; chunks with `start`, `end` and `tokens`, policies with `version` and `access` |
| Documents | 277; the last is H05, with its title, `version`, `access: 'public'` and `chunk_id` in the metadata |

---

## Part B: Build the embeddings from the settings

Part B gives LangChain the embedding model Lab 01 used.

LangChain calls an embeddings API through an **embeddings object**. `OpenAIEmbeddings`, from the `langchain-openai`
package, calls the same Embeddings API as Lab 01's client. `lc_embeddings()` in `shared/settings.py` builds it from
`models.env`, as `embed_client()` built Lab 01's client. Read it, then use it:

In [6]:
# embeddings: the LangChain object that embeds texts with EMBED_MODEL; Part C's store embeds with it
import inspect
from shared.settings import lc_embeddings

print(inspect.getsource(lc_embeddings))
embeddings = lc_embeddings(settings)
print("send token IDs:", embeddings.check_embedding_ctx_length)

def lc_embeddings(settings: dict | None = None):
    """The same embedding model as embed_client(), as LangChain OpenAIEmbeddings (Lab 05 on).

    LangChain splits long texts with tiktoken and sends token IDs by default; other OpenAI-compatible servers
    may reject token IDs, so that is switched off whenever the endpoint is not OpenAI's.
    """
    from langchain_openai import OpenAIEmbeddings
    settings = settings or load_settings()
    key = os.environ.get("OPENAI_API_KEY")
    if settings.get("EMBED_BASE_URL") and load_key("EMBED_API_KEY"):
        key = os.environ["EMBED_API_KEY"]
    return OpenAIEmbeddings(model=settings["EMBED_MODEL"], base_url=embed_url(settings), api_key=key,
                            check_embedding_ctx_length=embed_url(settings) == OPENAI_DEFAULT_URL)



send token IDs: True


`inspect.getsource()` prints a function's code. The `return` line builds `OpenAIEmbeddings` with four arguments:

- `model`: `EMBED_MODEL` from `models.env`, `text-embedding-3-small` here.
- `base_url`: the address Config printed as `embed_endpoint`.
- `api_key`: your key, or `EMBED_API_KEY` when `EMBED_BASE_URL` sends embeddings to a second provider (the `if` above
  it).
- `check_embedding_ctx_length`: when it is `True`, LangChain first cuts every text into **tokens** and sends their
  numbers, the **token IDs**, so that it can split a text longer than the model's limit. OpenAI accepts token IDs;
  many other OpenAI-compatible servers accept only text, so the helper turns it off for any endpoint that is not
  OpenAI's. Here it is `True`.

`embeddings` is the object Part C's store embeds with, and the labs from here on build it with `lc_embeddings()`.

### Checkpoint B

| You should see | Our run |
|---|---|
| `lc_embeddings()` | `send token IDs: True` on `https://api.openai.com/v1` |

### Check it against Lab 01's `embed()` (self-study)

The same model should give the same vector, whether LangChain or Lab 01's `embed()` sends the text. Embed the Tesla
question both ways:

In [7]:
# lc_vector: the question embedded through LangChain; plain_vector: the same question through Lab 01's embed()
import numpy as np
from shared.search import embed

question = "In what year was Nikola Tesla born?"
lc_vector = np.array(embeddings.embed_query(question))
plain_vector = embed([question], settings)[0]
print("numbers:", len(lc_vector), "| largest difference from embed():", f"{np.abs(lc_vector - plain_vector).max():.6f}")

numbers: 1536 | largest difference from embed(): 0.000000


1,536 numbers each, and the largest difference between matching numbers (`np.abs(a - b).max()`) is 0.000000: the
same vector. `embed_query()` embeds one text; `embed_documents()` embeds a list.

LangChain did not send your text, though. With `check_embedding_ctx_length` on, it cut the question into token IDs
with `tiktoken`, using the encoding tiktoken lists for the model (`cl100k_base` for OpenAI's embedding models), and sent
those. Make the same IDs yourself:

In [8]:
# token_ids: the question as the list of numbers LangChain sends in place of the text
import tiktoken

token_ids = tiktoken.get_encoding("cl100k_base").encode(question)
print("text:     ", question)
print("token IDs:", token_ids)

text:      In what year was Nikola Tesla born?
token IDs: [644, 1148, 1060, 574, 97415, 28298, 9405, 30]


Eight token IDs, one for each piece of the question. The API reads them as the same text, which is why the two
vectors above match. On an endpoint that is not OpenAI's, `lc_embeddings()` turns the switch off and LangChain sends
the text itself.

> A framework can send something other than what you wrote. **Check its result against your own on the same input:**
> here the request differs and the vector is the same.

| Self-study check | Our run |
|---|---|
| The vector | 1,536 numbers, largest difference from `embed()` 0.000000 |
| The token IDs | `[644, 1148, 1060, 574, 97415, 28298, 9405, 30]` |

---

## Part C: Index the documents and search with scores

Part C puts the 277 documents in a vector store and searches one question with both pipelines.

A **vector store** keeps each document with its vector and ranks the documents for a query.
[`InMemoryVectorStore`](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore),
from `langchain-core`, keeps them in a Python dictionary and ranks by cosine similarity: Lab 01's search behind
LangChain's interface. Stores backed by a database (pgvector, Qdrant and others) have the same methods.

Build the store with `embeddings` from Part B. `add_documents()` embeds every document and files it under its ID:

In [9]:
# store: the 277 documents with their vectors; ids: the ID each document was filed under, in order
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(embeddings)
ids = store.add_documents(documents)
print(len(ids), "documents stored | first IDs:", ids[:2], "| last:", ids[-2:])

277 documents stored | first IDs: ['S00-P00-C00', 'S00-P00-C01'] | last: ['H04', 'H05']


The store kept the documents' own IDs, from `S00-P00-C00` to `H05`.

Now search one question both ways: your `search()` from Lab 01, over the cached chunk vectors (embedded again if your
`.cache/` folder is empty), and the store's `similarity_search_with_score()`, called the way LangChain's
[semantic search tutorial](https://docs.langchain.com/oss/python/langchain/knowledge-base) calls it, with the question
alone.

> *In what year was Nikola Tesla born?*

*Predict first:* how many chunks will each search return?

In [10]:
# vectors: one cached vector per record; each search prints its hits as score and chunk ID, best first
from shared.search import chunk_vectors, search

question = "In what year was Nikola Tesla born?"
vectors = chunk_vectors(corpus, settings)
print("plain: search()")
for score, chunk in search(question, corpus, vectors, settings):
    print(f"  {score:.4f}  {chunk['chunk_id']}")
print("LangChain: store.similarity_search_with_score()")
for document, score in store.similarity_search_with_score(question):
    print(f"  {score:.4f}  {document.id}")

plain: search()


  0.7218  S03-P00-C00
  0.5407  S03-P01-C00
  0.4661  S03-P02-C00
LangChain: store.similarity_search_with_score()


  0.7221  S03-P00-C00
  0.5405  S03-P01-C00
  0.4656  S03-P02-C00
  0.2592  S03-P01-C01


- `search()` returns (score, chunk) pairs; the store returns (`Document`, score) pairs, the other way round.
- The same three chunks come first, in the same order, with scores less than a thousandth apart: the store embedded
  the chunks again, and two embeddings of the same text can differ by a few thousandths.
- **The store returned four rows where `search()` returned three.** Part D asks whether that changes the measured
  result.

This store's score is the cosine similarity, so higher is closer, as in Lab 01. Many stores backed by a database
return a **distance** instead, where lower is closer: check what a store's score means before you compare scores
across stores.

### Checkpoint C

| You should see | Our run |
|---|---|
| Stored | 277 documents, IDs from `S00-P00-C00` to `H05` |
| `search()` | `S03-P00-C00` about 0.72, `S03-P01-C00` 0.54, `S03-P02-C00` 0.47 |
| The store | the same three chunks with the same scores to two decimals, then a fourth row, `S03-P01-C01` about 0.26 |

---

## Part D: Check the two searches on 30 questions

Part D compares the two searches on the same 30 questions with the same measure. Checking a new implementation
against one you trust, on the same inputs, is a **parity check**.

The measure is Lab 01's **known-source hit@3**: a question is a hit when one of its top three chunks comes from a
paragraph in its `known_support`. Your pipeline has `hit_at_k()` for it. For the store, `is_hit()` reads each returned
document's `source_id` from its metadata:

In [11]:
# plain_hits: True or False per question from your pipeline; lc_results: the store's hits for each question
from shared.search import hit_at_k

questions = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
plain_hits = hit_at_k(questions, corpus, vectors, settings)

def is_hit(question, hits):
    return any(doc.metadata["source_id"] in question["known_support"] for doc, _ in hits)

lc_results = [store.similarity_search_with_score(q["question"]) for q in questions]
lc_hits = sum(is_hit(q, hits) for q, hits in zip(questions, lc_results))
print(f"hit@3: plain {sum(plain_hits)} of {len(questions)} | LangChain {lc_hits} of {len(questions)}")

hit@3: plain 30 of 30 | LangChain 30 of 30


### What happens step by step

1. `questions`: the 30 development questions, each with the paragraph IDs of its answer in `known_support`.
2. `hit_at_k()` embeds the 30 questions, ranks your cached vectors and returns one `True` or `False` per question.
3. `is_hit()` is `True` when any document the store returned has a `source_id` in the question's `known_support`.
4. The store searches every question the tutorial's way, with no `k`; `lc_results` keeps its hits for the next cells.
5. `sum()` counts the `True` values, since Python counts `True` as 1.

Both pipelines find a known paragraph for all 30 questions. Does that mean they return the same chunks? Compare what each returned, question by question: the same
chunk IDs in the same order.

*Predict first:* for how many of the 30 questions will the two lists be the same?

In [12]:
# plain_ids, lc_ids: the chunk IDs each pipeline returned, one list per question, best first
plain_results = [search(q["question"], corpus, vectors, settings) for q in questions]
plain_ids = [[chunk["chunk_id"] for _, chunk in hits] for hits in plain_results]
lc_ids = [[doc.id for doc, _ in hits] for hits in lc_results]
same = sum(a == b for a, b in zip(plain_ids, lc_ids))
print(f"same chunks in the same order: {same} of {len(questions)}")
print("first question | plain:", plain_ids[0], "| LangChain:", lc_ids[0])

same chunks in the same order: 0 of 30
first question | plain: ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00'] | LangChain: ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00', 'S00-P01-C00']


0 of 30. The first question shows why: the LangChain list has a fourth chunk. You never asked for four, so it
comes from a **default**, the value a function uses for an argument you do not pass. `inspect.signature()` prints a
function's arguments with their defaults:

In [13]:
# the arguments of the store's search method, with their default values
print(inspect.signature(store.similarity_search_with_score))

(query: 'str', k: 'int' = 4, **kwargs: 'Any') -> 'list[tuple[Document, float]]'


`k: int = 4`: the store returns four documents unless you pass `k`; your `search()` returns three. So the LangChain
"hit@3" above was really **hit@4**, and none of these 30 questions needed the fourth chunk, so the count could not
show the difference.

The answers would: the model reads a fourth chunk every time. Count the **evidence tokens**, the tokens of the chunks
an answer step would read, over the 30 questions, with four chunks and with the first three. `tiktoken` is the
tokenizer Lab 02 counted with, and `TOKEN_ENCODING` in `models.env` names its encoding:

In [14]:
# four, three: the tokens in the chunks returned for all 30 questions, with four chunks and with the first three
import tiktoken

encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])

def evidence_tokens(hits):
    return sum(len(encoding.encode(doc.page_content)) for doc, _ in hits)

four = sum(evidence_tokens(hits) for hits in lc_results)
three = sum(evidence_tokens(hits[:3]) for hits in lc_results)
print(f"evidence tokens over 30 questions: k=4 {four} | k=3 {three} | {four / three - 1:.0%} more")

evidence tokens over 30 questions: k=4 9946 | k=3 7764 | 28% more


About 9,900 tokens against 7,800: about 28% more input across the 30 answers, for the same hit@3 on this set. Pass `k=3` and compare the lists again:

In [15]:
# lc_results, lc_ids: the store's hits again, now with k=3; same: the questions whose two lists match
lc_results = [store.similarity_search_with_score(q["question"], k=3) for q in questions]
lc_ids = [[doc.id for doc, _ in hits] for hits in lc_results]
lc_hits = sum(is_hit(q, hits) for q, hits in zip(questions, lc_results))
same = sum(a == b for a, b in zip(plain_ids, lc_ids))
print(f"hit@3: LangChain {lc_hits} of {len(questions)} | same chunks in the same order: {same} of {len(questions)}")

hit@3: LangChain 30 of 30 | same chunks in the same order: 30 of 30


With `k=3` LangChain still scores 30 of 30, and the two lists match for every question. Last, the scores: the largest difference between the two pipelines' scores, rank by rank, over the 30
questions.

In [16]:
# gap: the largest difference between your score and the store's score at the same rank of the same question
from shared.settings import embed_url

gap = 0
for plain, lc in zip(plain_results, lc_results):
    for (plain_score, _), (_, lc_score) in zip(plain, lc):
        gap = max(gap, abs(plain_score - lc_score))
print(f"largest score difference: {gap:.4f}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

largest score difference: 0.0007

text-embedding-3-small at https://api.openai.com/v1


`(plain_score, _)` and `(_, lc_score)` take the score out of each pair: first in `search()`'s pairs, second in the
store's. The largest difference is under a thousandth, as two embeddings of the same text give. **With `k=3` the two searches match: the same chunks in the same order for every question.**

> Two habits come out of this part. **Compare two pipelines question by question, not by a total:** the total agreed
> before the lists did. And **pass every value your result depends on, even when the default looks harmless:** `k` is
> the number of chunks every answer reads.

### Checkpoint D

| You should see | Our run |
|---|---|
| hit@3 at the store's default | plain 30 of 30, LangChain 30 of 30 |
| Same lists | 0 of 30; the signature shows `k: int = 4` |
| Evidence tokens | k=4 9,946, k=3 7,764 (28% more) |
| With `k=3` | hit@3 30 of 30, same lists 30 of 30, largest score difference 0.0007 (a few thousandths at most) |

---

## Part E: Get a cited answer with `with_structured_output` (self-study)

Part E rebuilds Lab 02's answer step: the chat model, the answer's schema, and a reply your code can check.

`ChatOpenAI`, from `langchain-openai`, calls Chat Completions, the API Lab 02 called. Build it by hand from the
settings and send one message:

In [17]:
# chat: a LangChain chat model built by hand from the settings; reply: its answer to one message
import os
from langchain_openai import ChatOpenAI
from shared.settings import chat_url

chat = ChatOpenAI(model=settings["CHAT_MODEL"], base_url=chat_url(settings), api_key=os.environ["OPENAI_API_KEY"],
                  use_responses_api=False)
reply = chat.invoke("In one sentence, what does a vector store do?")
print(type(reply).__name__, "|", reply.content)
print(reply.usage_metadata)

AIMessage | A vector store is a database designed to efficiently store, index, and retrieve high-dimensional vectors, typically used for applications in machine learning and natural language processing, such as similarity search and recommendations.
{'input_tokens': 18, 'output_tokens': 38, 'total_tokens': 56, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


`invoke()` takes a string or a list of messages and returns an **`AIMessage`**, LangChain's object for a model's
reply:

- `content`: the reply text, Lab 02's `choices[0].message.content`.
- `usage_metadata`: the tokens billed, Lab 02's `usage`.

`use_responses_api=False` keeps `ChatOpenAI` on Chat Completions; left out, it may pick OpenAI's newer Responses API,
which other OpenAI-compatible servers usually do not offer.

This call sent no `temperature`. `lc_chat_model()` adds what `chat_options()` adds in Lab 02: `temperature`, or
`reasoning_effort` for a reasoning model. Read it, then use it:

In [18]:
# chat: the same model built by lc_chat_model(), with the options chat_options() picks for it
from shared.settings import lc_chat_model

print(inspect.getsource(lc_chat_model))
chat = lc_chat_model("CHAT_MODEL", settings)
print("model:", chat.model_name, "| temperature:", chat.temperature, "| reasoning_effort:", chat.reasoning_effort)

def lc_chat_model(name: str = "CHAT_MODEL", settings: dict | None = None, temperature: float = 0):
    """The same chat model as chat_client() + chat_options(), as a LangChain ChatOpenAI (Lab 05 on).

    use_responses_api=False keeps it on Chat Completions, which every compatible server implements.
    """
    from langchain_openai import ChatOpenAI
    settings = settings or load_settings()
    options = chat_options(name, settings, temperature)
    return ChatOpenAI(base_url=chat_url(settings), api_key=os.environ.get("OPENAI_API_KEY"),
                      use_responses_api=False, **options)

model: gpt-4o-mini | temperature: 0.0 | reasoning_effort: None


`temperature: 0.0` on `gpt-4o-mini`, as Lab 02 sent it; the labs build chat models with `lc_chat_model()` from here on.

Next, the schema. In Lab 02 you wrote it as a dictionary, `SCHEMA`, and checked each reply's shape with `parse()`.
LangChain's [structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output) takes a
**Pydantic** class instead: a Python class whose typed fields are the schema. Pydantic checks a reply when it builds
the object and raises an error when a field is missing or has the wrong type, so the class also does the work of
`parse()`. Write the answer's shape as two classes, one for a claim and one for the answer, and compare their required
fields with Lab 02's `SCHEMA`:

In [19]:
# Claim, CitedAnswer: the answer's shape as Pydantic classes; schema: the JSON schema Pydantic writes from them
from pydantic import BaseModel
from shared import answer as answering

class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str

class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]

schema = CitedAnswer.model_json_schema()
print("answer fields: Pydantic", schema["required"], "| Lab 02", answering.SCHEMA["required"])
print("claim fields:  Pydantic", schema["$defs"]["Claim"]["required"],
      "| Lab 02", answering.SCHEMA["properties"]["claims"]["items"]["required"])

answer fields: Pydantic ['answer', 'claims'] | Lab 02 ['answer', 'claims']
claim fields:  Pydantic ['claim', 'source_id', 'quote'] | Lab 02 ['claim', 'source_id', 'quote']


- `claims: list[Claim]`: a list of `Claim` objects, Lab 02's array of claims.
- `model_json_schema()`: the JSON schema Pydantic writes from the class. With `JSON_MODE=json_schema`, LangChain hands
  the class to the OpenAI SDK, which sends this schema, with `additionalProperties: false` added, as the reply format.
- The required fields are the same as Lab 02's.

The fields match; the text of the two schemas does not. Print the top-level keys and the `answer` field of each:

In [20]:
# the top-level keys of each schema, and the "answer" field as each schema writes it
print("Pydantic keys:", list(schema))
print("Lab 02 keys:  ", list(answering.SCHEMA))
print("Pydantic answer field:", schema["properties"]["answer"])
print("Lab 02 answer field:  ", answering.SCHEMA["properties"]["answer"])

Pydantic keys: ['$defs', 'properties', 'required', 'title', 'type']
Lab 02 keys:   ['type', 'properties', 'required', 'additionalProperties']
Pydantic answer field: {'title': 'Answer', 'type': 'string'}
Lab 02 answer field:   {'type': 'string'}


Pydantic puts the claim's fields under `$defs` and points to them from `claims`, adds a `title` to the class and to
every field, and orders the keys its own way; Lab 02's schema writes the claim in place, with no titles. **The model
reads the schema too**, so the same fields written another way can change a reply. Part F comes back to this.

The model reads evidence lines in Lab 02's format, `[ID] text`, and the quote check reads the same lines. Write them
from the store's hits, with `k=3`:

In [21]:
# evidence(): the hits as "[ID] text" lines; lines: the Tesla question's three lines
def evidence(hits):
    return [f"[{doc.id}] {doc.page_content}" for doc, _ in hits]

lines = evidence(store.similarity_search_with_score(question, k=3))
for line in lines:
    print(line[:90], "...")

[S03-P00-C00] Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) ...
[S03-P01-C00] Tesla gained experience in telephony and electrical engineering before emigr ...
[S03-P02-C00] Tesla went on to pursue his ideas of wireless lighting and electricity distr ...


`with_structured_output(CitedAnswer)` returns a new object built on the model. Its `method` follows `JSON_MODE`, as in
Lab 02: `json_schema` sends the schema and the API holds the reply to it (OpenAI calls this **Structured Outputs**);
`json_mode` only asks for JSON. Build it:

In [22]:
# structured: the chat model with CitedAnswer as its reply format; its replies come back as CitedAnswer objects
method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
structured = chat.with_structured_output(CitedAnswer, method=method)
print(type(structured).__name__, "| method:", method)

RunnableSequence | method: json_schema


A **`RunnableSequence`**, LangChain's name for steps run one after another: first the model, with `CitedAnswer` sent as
the reply format on every call; then a parser that builds a `CitedAnswer` from the reply. Ask the Tesla question with
Lab 02's rules, `shared.answer.RULES` (abstention line included), as a system message and a user message:

In [23]:
# messages_for(): Lab 02's system and user messages; reply: the Tesla answer as a CitedAnswer object
def messages_for(question, lines):
    return [("system", answering.RULES), ("user", f"Question: {question}\nEvidence:\n" + "\n".join(lines))]

reply = structured.invoke(messages_for(question, lines))
print(type(reply).__name__, "|", reply.answer)
for claim in reply.claims:
    print(f"  quote found: {answering.quote_found(claim.model_dump(), lines)} | {claim.source_id} | {claim.quote}")

CitedAnswer | Nikola Tesla was born in the year 1856.
  quote found: True | S03-P00-C00 | 10 July 1856


The reply is a `CitedAnswer` object, not text: *Nikola Tesla was born in the year 1856.*, with one claim from `S03-P00-C00` (*10 July 1856*) whose quote is found. `claim.model_dump()` turns a claim into the dictionary Lab 02's `quote_found()` reads, so the check is
unchanged.

> **`with_structured_output` replaces Lab 02's `response_format`, the JSON parsing and `parse()` with one object;** the
> quote check stays yours.

### Checkpoint E

| You should see | Our run |
|---|---|
| Hand-built model | an `AIMessage`, with `usage_metadata` |
| `lc_chat_model()` | `temperature: 0.0`, `reasoning_effort: None` on `gpt-4o-mini` |
| Schema | the same required fields as Lab 02's `SCHEMA`; Pydantic's keys include `$defs` and `title` |
| `structured` | `RunnableSequence`, method `json_schema` |
| Tesla | *Nikola Tesla was born in the year 1856.* (or *on 10 July 1856*), quote found: `True` |

---

## Part F: Check the two answer steps on 30 questions (self-study)

Part F compares the two answer steps on all 30 development questions, one answer each. Both get the same evidence
lines, the store's three chunks for each question, so any difference comes from the answer step alone. These are not
the lines Labs 02 and 04 gave the model (`pack()` over five chunks, inside 400 tokens); Part H counts that difference.

In [24]:
# all_lines: each question's evidence lines; plain_replies, lc_replies: one answer per question from each pipeline
INSUFFICIENT = "Insufficient evidence."
all_lines, plain_replies, lc_replies = [], [], []
for q in questions:
    q_lines = evidence(store.similarity_search_with_score(q["question"], k=3))
    all_lines.append(q_lines)
    plain_replies.append(answering.answer(q["question"], q_lines, settings))
    lc_replies.append(structured.invoke(messages_for(q["question"], q_lines)).model_dump())
print(len(plain_replies), "answers from each pipeline")

30 answers from each pipeline


`model_dump()` turned each LangChain answer into the dictionary Lab 02's `answer()` returns, so one function can count
both. It counts Lab 02's measures: refusals (these 30 questions should get none), claims and quotes found.

In [25]:
# summary(): refusals, claims and quotes found over 30 replies; cited(): the source IDs one reply cites, in order
def summary(replies):
    refused = sum(reply["answer"].strip() == INSUFFICIENT for reply in replies)
    claims = [(q_lines, claim) for q_lines, reply in zip(all_lines, replies) for claim in reply["claims"]]
    found = sum(answering.quote_found(claim, q_lines) for q_lines, claim in claims)
    return f"refused {refused} of {len(replies)} | claims {len(claims)} | quotes found {found}"

def cited(reply):
    return [claim["source_id"] for claim in reply["claims"]]

print("plain:    ", summary(plain_replies))
print("LangChain:", summary(lc_replies))
print("same cited IDs:", sum(cited(p) == cited(l) for p, l in zip(plain_replies, lc_replies)), "of 30",
      "| same answer text:", sum(p["answer"] == l["answer"] for p, l in zip(plain_replies, lc_replies)), "of 30")

plain:     refused 0 of 30 | claims 31 | quotes found 31
LangChain: refused 0 of 30 | claims 32 | quotes found 32
same cited IDs: 29 of 30 | same answer text: 29 of 30


### What happens step by step

1. `summary()` counts the replies whose answer is *Insufficient evidence.*
2. It pairs every claim with its own question's evidence lines and counts the claims.
3. `quote_found()` checks each claim's quote against those lines and the found ones are counted.
4. `cited()` lists a reply's cited IDs in order, so two replies can be compared with `==`.

Neither pipeline refuses, and every quote is found in both: 31 claims plain, 32 LangChain in our run. The cited IDs agree for 29 of 30 questions, and so does the answer text. Print the answers whose text or cited IDs differ, or that have a quote the check did not find (printed
after the answer):

In [26]:
# not_found(): the quotes of one reply that the check did not find in its lines; the loop prints every pair that differs
def not_found(reply, q_lines):
    return [claim["quote"] for claim in reply["claims"] if not answering.quote_found(claim, q_lines)]

for plain, lc, q_lines in zip(plain_replies, lc_replies, all_lines):
    if plain["answer"] != lc["answer"] or cited(plain) != cited(lc) or not_found(plain, q_lines) + not_found(lc, q_lines):
        print("plain:    ", cited(plain), plain["answer"], not_found(plain, q_lines) or "")
        print("LangChain:", cited(lc), lc["answer"], not_found(lc, q_lines) or "", "\n")

plain:     ['S01-P00-C01'] 1745 
LangChain: ['S01-P00-C01'] Casimir Pulaski was born in Warsaw in 1745.  

plain:     ['S02-P02-C00'] The original meaning of the word Norman is "Northman" or "Norseman, Viking." 
LangChain: ['S02-P02-C00', 'S02-P00-C00'] The original meaning of the word Norman is "Northman" or "Norseman, Viking."  



Two questions differ. For Pulaski the two answers state the same fact from the same chunk in other words: *1745* against *Casimir Pulaski was born in Warsaw in 1745.* For the Normans the text is the same and LangChain also cites a second chunk, `S02-P00-C00`. Are these differences the framework, or the model? A model can word the same answer differently from
one call to the next, so compare against a **control**: ask Lab 02's `answer()` the same 30 questions once more and
compare it with its own first run.

In [27]:
# again: a second run of your answer step on the same lines, the control
again = [answering.answer(q["question"], q_lines, settings) for q, q_lines in zip(questions, all_lines)]
print("plain, second run:", summary(again))
print("same answer text as the first run:", sum(a["answer"] == p["answer"] for a, p in zip(again, plain_replies)), "of 30")

plain, second run: refused 0 of 30 | claims 31 | quotes found 31
same answer text as the first run: 30 of 30


Against its own first run, the plain pipeline gives the same text for all 30 questions in our run, with 31 claims, all found. The two pipelines differ a little more than one pipeline differs from itself (29 against 30 of 30), in wording and one extra citation.

One difference may not show in a single run. Lab 02's `quote_found()` accepts a quote cut with `...` when both parts
are in the line, in order, and LangChain's answer step cuts quotes that way more often. **When we ran it**
(`gpt-4o-mini`) on *How many fumbles did Von Miller force in Super Bowl 50?*, 11 of 20 calls with the Pydantic schema
cut the quote, against 0 of 20 with Lab 02's schema. The messages were the same, so the schema made the difference:
Part E's `$defs`, titles and key order are part of what the model reads.

Last, the question Lab 02 used to test abstention, which the corpus cannot answer, three times in each pipeline:

> *Who was the first person to walk on the Moon?*

In [28]:
# plain_moon, lc_moon: three answers each to a question the corpus cannot answer
moon = "Who was the first person to walk on the Moon?"
moon_lines = evidence(store.similarity_search_with_score(moon, k=3))
plain_moon = [answering.answer(moon, moon_lines, settings)["answer"] for _ in range(3)]
lc_moon = [structured.invoke(messages_for(moon, moon_lines)).answer for _ in range(3)]
print(f"plain     abstained {plain_moon.count(INSUFFICIENT)} of 3 | {plain_moon[0]}")
print(f"LangChain abstained {lc_moon.count(INSUFFICIENT)} of 3 | {lc_moon[0]}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

plain     abstained 3 of 3 | Insufficient evidence.
LangChain abstained 3 of 3 | Insufficient evidence.

gpt-4o-mini at https://api.openai.com/v1


Both abstain in 3 of 3. The comparison for the whole pipeline, from this run:

| Measure | Plain Python (Labs 01 and 02) | LangChain |
|---|---|---|
| Known-source hit@3 on the 30 questions | 30 | 30 with `k=3` |
| Same chunks in the same order | (the reference) | 30 of 30 with `k=3`; 0 of 30 at the default `k=4` |
| Refused, of 30 answerable questions | 0 | 0 |
| Claims, and quotes found | 31, 31 | 32, 32 |
| The Moon question, abstained | 3 of 3 | 3 of 3 |

The framework changed the code: a `Document` for a dictionary, a store for the NumPy product, a Pydantic class for the
schema and `parse()`. The comparison found two differences that came with it: the store's default `k=4`, and the
schema LangChain writes from the Pydantic class.

> **Given the same evidence lines, the two answer steps agree on refusals and quotes found;** to judge their wording,
> compare the two pipelines against a second run of one of them, the control.

### Checkpoint F

| You should see | Our run |
|---|---|
| 30 questions | both refuse 0 of 30, every quote found (31 and 32 claims); same cited IDs 29 of 30, same text 29 of 30 |
| Plain, second run | 0 refused, 31 claims all found; same text as its first run 30 of 30 |
| The Moon question | *Insufficient evidence.* 3 of 3 (plain), 3 of 3 (LangChain) |

---

## Part G: Leave documents out with a metadata filter (self-study)

Part G uses the metadata kept in Part A: the store can leave documents out before it ranks.

A customer who uses Atlas asks:

> *How quickly do reset links expire?*

The question names no product. *Predict first:* which product's reset time will the answer give? Search it and answer
it three times:

In [29]:
# reset_lines: the evidence lines for the question; the loop asks the LangChain answer step three times
reset = "How quickly do reset links expire?"
reset_lines = evidence(store.similarity_search_with_score(reset, k=3))
print("evidence:", [line[1:line.index("]")] for line in reset_lines])
for _ in range(3):
    print(" ", structured.invoke(messages_for(reset, reset_lines)).answer)

evidence: ['H02', 'H04', 'H01']


  Atlas reset links expire after 30 minutes and Boreal reset links expire after 10 minutes.


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


`line[1:line.index("]")]` reads the ID between the brackets of each line. H02 comes first, then H04, the Boreal policy, and all three answers give both products' times: *Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.* For an Atlas customer the
Boreal policy is noise, and the question's words cannot keep it out: only the customer's account says which product
they use.

A **metadata filter** can. In `InMemoryVectorStore` a filter is a function that takes a `Document` and returns `True`
to keep it, and the store applies it to every document before it ranks. Each policy's title starts with its product's
name, and no chunk title starts with *Atlas*:

In [30]:
# atlas_only(): True for the Atlas policies; atlas_lines: the evidence once every other document is left out
def atlas_only(document):
    return document.metadata["title"].startswith("Atlas")

atlas_lines = evidence(store.similarity_search_with_score(reset, k=3, filter=atlas_only))
print("evidence:", [line[1:line.index("]")] for line in atlas_lines])
for _ in range(3):
    print(" ", structured.invoke(messages_for(reset, atlas_lines)).answer)

evidence: ['H02', 'H01', 'H05']


  Atlas reset links expire after 30 minutes.


  Atlas reset links expire after 30 minutes.


  Atlas reset links expire after 30 minutes.


H04 is gone and the Atlas billing policy H05 takes third place; all three answers give Atlas's 30 minutes only. **The filter chose by who is asking, which a similarity score cannot.** Lab 11 uses the same
mechanism for permissions, with the `access` field. A store that keeps its data in a database takes the filter in its
own syntax, often a dictionary, because the database applies it.

### Checkpoint G

| You should see | Our run |
|---|---|
| No filter | evidence H02, H04, H01; both products' times in 3 of 3 |
| `atlas_only` | evidence H02, H01, H05; *Atlas reset links expire after 30 minutes.* in 3 of 3 |

---

## Part H: Keep the pipeline in `shared/lc_pipeline.py` (self-study)

Part H moves the LangChain pipeline into one module, the baseline the next labs measure their changes against.

`shared/lc_pipeline.py` holds the steps of Parts A, C and E, with `k=3` set in the search and `JSON_MODE` read from
the settings. Its `load_corpus()` is Lab 04's `corpus()` from `shared/trace.py`, the same 277 records as Part A,
imported under the name the next labs use, so the corpus is defined in one place. Read it:

In [31]:
# the whole module: the classes of Part E and one function for each step
import shared.lc_pipeline

print(inspect.getsource(shared.lc_pipeline))

"""The pipeline of Labs 01 and 02, rebuilt with LangChain in Lab 05: the baseline for the labs after it.

load_corpus() is Lab 04's corpus(): the 272 chunks and the five policies. to_documents() turns its records into
LangChain Documents. build_store() embeds them into an InMemoryVectorStore. retrieve() searches it, with k=3 by
default (the store's own default is 4). evidence() writes the hits as "[ID] text" lines, and answer() asks for a
CitedAnswer with Lab 02's rules, so shared.answer.quote_found() checks its claims unchanged.
"""
from __future__ import annotations

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from pydantic import BaseModel

from shared.answer import RULES
from shared.settings import lc_chat_model, lc_embeddings
from shared.trace import corpus as load_corpus


class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str


class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]


d

Use it from the start, as a later lab will: corpus, documents and store, then search, evidence lines, answer and the
quote check, for the Tesla and Moon questions.

In [32]:
# pipeline_store: a new store built by the module; each question goes through retrieve(), evidence() and answer()
from shared import lc_pipeline

pipeline_store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
for q in (question, moon):
    q_lines = lc_pipeline.evidence(lc_pipeline.retrieve(pipeline_store, q))
    reply = lc_pipeline.answer(q, q_lines, settings)
    print(reply.answer, "| quotes found:", [answering.quote_found(claim.model_dump(), q_lines) for claim in reply.claims])

Nikola Tesla was born in the year 1856. | quotes found: [True]


Insufficient evidence. | quotes found: []


The module gives the same results: Tesla in 1856 with its quote found, and *Insufficient evidence.* for the Moon.

One difference is left, in what the answer step is given. The pipeline you traced in Lab 04 searches five chunks and
packs those that fit in 400 tokens (`pack()`, Lab 02); `lc_pipeline` keeps three chunks with no budget (`evidence()`).
Part F gave both answer steps the same lines, so it could not see this. Count the lines and tokens each one gives the
model for the 30 questions, with `tiktoken` (no chat call):

In [33]:
# packed: pack()'s lines per question (k=5, 400 tokens); lc_lines: evidence()'s lines per question (k=3, no budget)
def tokens(lines):
    return sum(len(encoding.encode(line)) for line in lines)

packed = [answering.pack(search(q["question"], corpus, vectors, settings, k=5), settings) for q in questions]
lc_lines = [lc_pipeline.evidence(lc_pipeline.retrieve(pipeline_store, q["question"])) for q in questions]
for name, per_question in [("pack()", packed), ("evidence()", lc_lines)]:
    counts = [len(q_lines) for q_lines in per_question]
    print(f"{name:11} {sum(counts)} lines, {sum(tokens(q_lines) for q_lines in per_question)} tokens"
          f" | lines a question: {sorted(set(counts))}")
print("questions where pack() gives more lines:", sum(len(a) > len(b) for a, b in zip(packed, lc_lines)), "of", len(questions))

pack()      125 lines, 10761 tokens | lines a question: [3, 4, 5]
evidence()  90 lines, 8382 tokens | lines a question: [3]
questions where pack() gives more lines: 23 of 30


`pack()` gives the model 125 lines and about 10,800 tokens over the 30 questions, three to five lines a question; `evidence()` gives 90 lines and about 8,400 tokens, three for every question. For 23 of 30 questions the answer step in Labs 06 to 11 reads one or two chunks fewer than the one you traced in Lab 04. Both rank by the same scores (Part D), so the three lines are the first three of the pack. **When an
answer in Lab 06 differs from Lab 04's, the framework is not the only change: the evidence changed too.** To compare two
answer steps, give them the same lines, as Part F did.

From here on the labs call `load_corpus()`, `to_documents()`, `build_store()`, `retrieve()`, `evidence()` and
`answer()` by name, and keep Lab 02's `quote_found()` for the check.

### Checkpoint H

| You should see | Our run |
|---|---|
| Module results | *Nikola Tesla was born in the year 1856.*, quote found; *Insufficient evidence.* for the Moon |
| Evidence given | `pack()` 125 lines, 10,761 tokens (3 to 5 a question); `evidence()` 90 lines, 8,382 tokens; more lines from `pack()` for 23 of 30 |

---

## What you can do now

- Turn chunk records into `Document`s that keep their IDs and fields as metadata.
- Build LangChain's embeddings and chat model from `models.env` with `lc_embeddings()` and `lc_chat_model()`, and say
  why the embeddings send token IDs to OpenAI and text to other servers.
- Index documents in `InMemoryVectorStore` and search them with scores.
- Compare a framework's results with your own question by question, and find a default in a function's signature.
- Get a cited answer as a Pydantic object with `with_structured_output`, check its quotes with `quote_found()`, and
  explain why its answers can differ from Lab 02's: wording that varies between runs, the schema LangChain writes, and
  fewer evidence lines than `pack()` gave in Labs 02 and 04.
- Leave documents out of a search with a metadata filter.

## Challenge

Use only what this lab showed. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Check a retriever against your own search.** Much LangChain code searches through a **retriever**, an object with
   `invoke()` and `batch()` that returns documents without scores. `store.as_retriever()` makes one from the store.
   Does it return what your `search()` returns?

   - Under **Task 1**, make a retriever with `store.as_retriever()`. The cell runs it on the 30 questions and counts
     the questions whose chunk IDs match `plain_ids` from Part D, in order.
   - If they do not all match, find out why from what the cell prints and the
     [`as_retriever` reference](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore),
     then fix the retriever and run the cell again.

   **Check:** the count before and after your fix. Our worked solution got 0 of 30, then 30 of 30.

   **Explain:** which check in this lab would have missed the difference, and which one caught it?

2. **Optional: write a filter for another customer.** Complete `boreal_only` under **Task 2** for a customer who uses
   Boreal. The cell asks Part G's question and one the Boreal policy does not answer, *How are Boreal invoices sent?*

   **Check:** for each question, the evidence IDs and the answer. Our worked solution got `['H04']` and Boreal's 10 minutes (*Reset links expire after 10 minutes.*) for Part G's question, and `['H04']` and *Insufficient evidence.* for the invoices question.

   **Explain:** how many documents did the store return for each question, and why? What would the model have had to
   go on without the abstention line?

**Task 1.** Your retriever, then the count:

In [34]:
# retriever: your retriever from the store; the cell counts the questions whose IDs match plain_ids from Part D
retriever = None    # TODO: a retriever from store.as_retriever()

if retriever is None:
    print("⚠️ Make a retriever, then run the cell again.")
else:
    retriever_ids = [[doc.id for doc in docs] for docs in retriever.batch([q["question"] for q in questions])]
    print(f"same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, retriever_ids))} of {len(questions)}")
    print("chunks per question:", len(retriever_ids[0]), "| search_kwargs:", retriever.search_kwargs)

⚠️ Make a retriever, then run the cell again.


**Task 2 (optional).** Complete `boreal_only`, then run the cell:

In [35]:
# boreal_only(): True for the Boreal documents; the cell asks two questions with it as the filter
def boreal_only(document):
    # TODO: return True for documents about Boreal (see atlas_only in Part G)
    return None

if boreal_only(documents[-1]) is None:
    print("⚠️ Complete boreal_only, then run the cell again.")
else:
    for q in (reset, "How are Boreal invoices sent?"):
        q_lines = evidence(store.similarity_search_with_score(q, k=3, filter=boreal_only))
        print([line[1:line.index("]")] for line in q_lines], "|", structured.invoke(messages_for(q, q_lines)).answer)

⚠️ Complete boreal_only, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'langchain_community'` | code from an older tutorial; these labs install only LangChain's core packages | Import the store from `langchain_core.vectorstores` and the models from `langchain_openai`, as this lab does |
| `OpenAIError: Missing credentials. Please pass an api_key ...` | no key in the environment when the model or embeddings object was built | Add the key as Lab 00's README shows for your route, run Config, then build the object again |
| `OpenAIInvalidRequestError: Error code: 400 ... 'temperature' does not support 0.0 with this model.` | `ChatOpenAI(temperature=0)` built by hand for a reasoning model such as `gpt-6-luna` | Build models with `lc_chat_model()`, and set `REASONING_EFFORT=none` for `gpt-6-luna` |
| `ValueError: Unsupported function` followed by a schema | Lab 02's `SCHEMA` dictionary passed to `with_structured_output`, which needs a name for a dictionary schema | Pass the `CitedAnswer` class, or `{"name": "cited_answer", "strict": True, "schema": SCHEMA}` |
| `KeyError: 'access'` during a filtered search | the filter reads a field only some documents have (only the policies have `access`) | Read it with `document.metadata.get("access")` |

## Resources

- [LangChain: build a semantic search engine](https://docs.langchain.com/oss/python/langchain/knowledge-base): documents,
  embeddings, vector stores and retrievers, the steps of Parts A to D
- [LangChain: models, structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output):
  `with_structured_output` with a Pydantic class, its methods, and `include_raw=True` for the token counts
- [LangChain reference: `InMemoryVectorStore`](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore):
  every search method, its defaults and the filter argument
- [LangChain reference: `OpenAIEmbeddings`](https://reference.langchain.com/python/langchain-openai/embeddings/base/OpenAIEmbeddings):
  `check_embedding_ctx_length` and the settings for other OpenAI-compatible servers
- [Pydantic: models](https://docs.pydantic.dev/latest/concepts/models/): fields, validation and `model_dump()`

Next: [Lab 06](../06_Evaluation_Set/README.md), *How to build an evaluation set for RAG*. Back to [the lab list](../README.md).